In [ ]:
# In[1]:

import pandas as pd

# Load telemetry CSVs (timestamps not converted as requested)
metric = pd.read_csv("metric.csv")
log = pd.read_csv("log.csv")
trace = pd.read_csv("trace.csv")

# 1) metric.csv aggregation across entire file: count and global P95 per (cmdb_id, kpi_name)
metric_summary = (
    metric
    .groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(
        count=("value", "count"),
        p95_value=("value", lambda x: float(x.quantile(0.95)))
    )
    .sort_values("count", ascending=False)
)
metric_summary_top20 = metric_summary[["cmdb_id", "kpi_name", "count", "p95_value"]].head(20)

# 2) unique cmdb_id list (short) and unique kpi_name list (limit to first 50)
metric_cmdb_ids = pd.Series(sorted(metric["cmdb_id"].unique()), name="cmdb_id")
metric_kpi_names = pd.Series(sorted(metric["kpi_name"].unique())[:50], name="kpi_name")

# 3) log.csv unique (cmdb_id, log_name) pairs with counts, sorted desc, up to 20 rows
log_pairs = (
    log
    .groupby(["cmdb_id", "log_name"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
)
log_pairs_top20 = log_pairs.head(20)

# 4) trace.csv unique (cmdb_id, trace_name) pairs with counts, sorted desc, up to 20 rows
trace_pairs = (
    trace
    .groupby(["cmdb_id", "trace_name"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
)
trace_pairs_top20 = trace_pairs.head(20)

# Display compact results (use variable names as outputs)
metric_summary_top20, metric_cmdb_ids, metric_kpi_names, log_pairs_top20, trace_pairs_top20

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregation:\n"
    "- Metrics: Grouping by (cmdb_id, kpi_name) returned up to 20 groups; most groups have 25 samples each.\n"
    "- Notable P95 values (from metric groups):\n"
    "  * Highest memory (p95): catalogue-db = 4.225575e+08, carts = 2.079674e+08.\n"
    "  * Highest CPU (p95): carts-db = 5.597895, carts = 2.035317.\n"
    "  * Highest disk I/O (p95): carts-db = 2.429998e+06.\n"
    "  * Socket counts: carts socket p95 = 14, carts-db/catalogue socket = 9.\n"
    "- Unique metric cmdb_id values (15 services): ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db'].\n"
    "- Unique metric kpi_name values (7 shown): ['cpu', 'diskio', 'latency-50', 'latency-90', 'mem', 'socket', 'workload'].\n"
    "- Logs: unique (cmdb_id, log_name) pairs show log.error_count and log.row_count for many services (counts mostly 25). carts has 20 samples for logs, carts-db has 24, user-db has 6.\n"
    "- Traces: no trace records were found (trace pairs result is empty).\n"
    "Overall: metrics are present for many services with consistent sample counts; the most prominent resource anomalies by P95 are high memory on catalogue-db and high disk I/O and CPU on carts-db."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count     p95_value
0          carts         cpu     25  2.035317e+00
1          carts  latency-50     25  1.922075e-02
2          carts  latency-90     25  4.223964e-02
3          carts         mem     25  2.079674e+08
4          carts      socket     25  1.400000e+01
5          carts    workload     25  8.569177e+00
6       carts-db         cpu     25  5.597895e+00
7       carts-db      diskio     25  2.429998e+06
8       carts-db         mem     25  9.398987e+07
9       carts-db      socket     25  9.000000e+00
10     catalogue         cpu     25  1.871834e-01
11     catalogue  latency-50     25  3.150474e-03
12     catalogue  latency-90     25  4.870852e-03
13     catalogue         mem     25  6.290855e+06
14     catalogue      socket     25  9.000000e+00
15     catalogue    workload     25  4.315570e+00
16  catalogue-db         cpu     25  2.442308e-01
17  catalogue-db      diskio     25  0.000000e+00
18  catalogue-db         mem     25  4.225575e+08
19  catalogue-db      socket     25  4.000000e+00, 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object, 0           cpu
1        diskio
2    latency-50
3    latency-90
4           mem
5        socket
6      workload
Name: kpi_name, dtype: object,          cmdb_id         log_name  count
4      catalogue  log.error_count     25
5      catalogue    log.row_count     25
9         orders    log.row_count     25
8         orders  log.error_count     25
7      front-end    log.row_count     25
6      front-end  log.error_count     25
17      shipping    log.row_count     25
16      shipping  log.error_count     25
19          user    log.row_count     25
18          user  log.error_count     25
10       payment  log.error_count     25
11       payment    log.row_count     25
12  queue-master  log.error_count     25
13  queue-master    log.row_count     25
2       carts-db  log.error_count     24
3       carts-db    log.row_count     24
0          carts  log.error_count     20
1          carts    log.row_count     20
20       user-db  log.error_count      6
21       user-db    log.row_count      6, Empty DataFrame
Columns: [cmdb_id, trace_name, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing 'metric' dataframe loaded previously; if not present, load it
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# 1) Compute global P95 and P90 per (cmdb_id, kpi_name) using the entire series (before any time filtering)
thresholds = (
    metric
    .groupby(["cmdb_id", "kpi_name"], as_index=False)["value"]
    .agg(p95=lambda x: float(x.quantile(0.95)), p90=lambda x: float(x.quantile(0.90)))
)

# 2) Parse timestamps (UTC) and filter to the incident window (inclusive)
metric["timestamp_dt"] = pd.to_datetime(metric["timestamp"], unit="s", utc=True)
window_start = pd.Timestamp("2024-01-20 21:28:56", tz="UTC")
window_end = pd.Timestamp("2024-01-20 21:58:56", tz="UTC")
metric_window = metric[metric["timestamp_dt"].between(window_start, window_end)]

# function to compute anomaly summary given a threshold column name ('p95' or 'p90')
def compute_anomalies(threshold_col):
    # merge threshold into windowed points
    mw = metric_window.merge(thresholds[["cmdb_id", "kpi_name", threshold_col]],
                              on=["cmdb_id", "kpi_name"], how="left")
    if mw.empty:
        return pd.DataFrame([], columns=[
            "cmdb_id", "kpi_name", "global_threshold", "anomaly_count_in_window",
            "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
        ])
    # anomalies based on threshold
    anomalies = mw[mw["value"] >= mw[threshold_col]]
    if anomalies.empty:
        return pd.DataFrame([], columns=[
            "cmdb_id", "kpi_name", "global_threshold", "anomaly_count_in_window",
            "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
        ])
    anomalies_grp = (
        anomalies
        .groupby(["cmdb_id", "kpi_name"], as_index=False)
        .agg(
            anomaly_count_in_window=("value", "count"),
            earliest_anomaly_timestamp_utc=("timestamp_dt", "min"),
        )
    )
    # window stats (max and median among all points in the window for the group)
    window_stats = (
        mw
        .groupby(["cmdb_id", "kpi_name"], as_index=False)
        .agg(
            max_value_in_window=("value", "max"),
            median_value_in_window=("value", "median"),
        )
    )
    # merge and attach global threshold
    result = (
        anomalies_grp
        .merge(window_stats, on=["cmdb_id", "kpi_name"], how="left")
        .merge(thresholds[["cmdb_id", "kpi_name", threshold_col]], on=["cmdb_id", "kpi_name"], how="left")
    )
    # rename columns to match spec
    result = result.rename(columns={threshold_col: "global_threshold"})
    # format earliest timestamp as ISO string
    result["earliest_anomaly_timestamp_utc"] = result["earliest_anomaly_timestamp_utc"].apply(lambda x: x.isoformat())
    # ensure ordering of columns
    result = result[[
        "cmdb_id", "kpi_name", "global_threshold", "anomaly_count_in_window",
        "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
    ]]
    return result

# 3) Try P95 first
result_p95 = compute_anomalies("p95")

if not result_p95.empty:
    # compute severity for sorting and sort as required, limit to top 20
    result_p95["severity"] = result_p95["max_value_in_window"] - result_p95["global_threshold"]
    result_df = result_p95.sort_values(["anomaly_count_in_window", "severity"], ascending=[False, False]).drop(columns=["severity"]).head(20)
    used_threshold = "P95"
else:
    # fallback to P90 if no P95 breaches
    result_p90 = compute_anomalies("p90")
    if not result_p90.empty:
        result_p90["severity"] = result_p90["max_value_in_window"] - result_p90["global_threshold"]
        result_df = result_p90.sort_values(["anomaly_count_in_window", "severity"], ascending=[False, False]).drop(columns=["severity"]).head(20)
        used_threshold = "P90"
    else:
        # no breaches for P95 or P90 -> empty result
        result_df = pd.DataFrame([], columns=[
            "cmdb_id", "kpi_name", "global_threshold", "anomaly_count_in_window",
            "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
        ])
        used_threshold = "none (no P95 or P90 breaches in window)"

# Display compact result (up to 20 rows) and which threshold was used
result_df, used_threshold

```
Out[2]:
```
Summary (threshold used: global P95)

- P95 breaches were detected in the incident window (2024-01-20 21:28:56 → 21:58:56 UTC). Results limited to top groups by anomaly count:

Top groups (compact):
- catalogue-db / diskio — global_p95=0.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=0.0 — median=0.0  
- catalogue-db / socket — global_p95=4.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=4.0 — median=4.0  
- orders-db / socket — global_p95=6.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=6.0 — median=6.0  
- payment / socket — global_p95=6.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=6.0 — median=6.0  
- queue-master / socket — global_p95=3.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=3.0 — median=3.0  
- rabbitmq / diskio — global_p95=0.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=0.0 — median=0.0  
- rabbitmq / socket — global_p95=11.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=11.0 — median=11.0  
- rabbitmq-exporter / mem — global_p95≈4.72e6 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max≈4.72e6 — median≈4.72e6  
- rabbitmq-exporter / socket — global_p95=1.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=1.0 — median=1.0  
- session-db / socket — global_p95=3.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=3.0 — median=3.0  
- user-db / socket — global_p95=18.0 — anomalies=25 — earliest=2024-01-20T21:31:00Z — max=18.0 — median=18.0  
- carts-db / socket — global_p95=9.0 — anomalies=20 — earliest=2024-01-20T21:31:00Z — max=9.0 — median=9.0  
- user / socket — global_p95=37.0 — anomalies=11 — earliest=2024-01-20T21:45:00Z — max=37.0 — median=34.0  
- catalogue / socket — global_p95=9.0 — anomalies=5 — earliest=2024-01-20T21:31:00Z — max=9.4 — median=8.0  
- carts / socket — global_p95=14.0 — anomalies=3 — earliest=2024-01-20T21:31:00Z — max=14.0 — median≈13.2667  

Notable smaller-count but large-value anomalies:
- user / diskio — global_p95≈1.768e10 — anomalies=2 — earliest=2024-01-20T21:51:00Z — max≈1.790e10 — median≈1.48e5  
- user / mem — global_p95≈7.655e8 — anomalies=2 — earliest=2024-01-20T21:44:00Z — max≈8.388e8 — median≈9.54e6  
- front-end / mem — global_p95≈1.064e8 — anomalies=2 — earliest=2024-01-20T21:32:00Z — max≈1.071e8 — median≈1.037e8  
- shipping / mem — global_p95≈3.026e8 — anomalies=2 — earliest=2024-01-20T21:39:00Z — max≈3.032e8 — median≈3.019e8  
- user-db / mem — global_p95≈6.041e7 — anomalies=2 — earliest=2024-01-20T21:54:00Z — max≈6.073e7 — median≈5.797e7

Observations:
- Many services show socket-related values breaching P95 throughout the window (several with anomaly_count=25). Most of these have earliest anomaly at ~2024-01-20T21:31:00Z.
- Memory and disk I/O anomalies are present for a few services (notably rabbitmq-exporter and user), though some diskio/memory P95 thresholds are zero or the values equal the threshold (leading to equality-based breaches).
- Overall, P95 breaches are concentrated in socket metrics across many components, plus isolated high mem/diskio values for user and rabbitmq-exporter.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  global_threshold  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  median_value_in_window
17       catalogue-db   diskio      0.000000e+00                       25      2024-01-20T21:31:00+00:00         0.000000e+00            0.000000e+00
19       catalogue-db   socket      4.000000e+00                       25      2024-01-20T21:31:00+00:00         4.000000e+00            4.000000e+00
35          orders-db   socket      6.000000e+00                       25      2024-01-20T21:31:00+00:00         6.000000e+00            6.000000e+00
40            payment   socket      6.000000e+00                       25      2024-01-20T21:31:00+00:00         6.000000e+00            6.000000e+00
45       queue-master   socket      3.000000e+00                       25      2024-01-20T21:31:00+00:00         3.000000e+00            3.000000e+00
47           rabbitmq   diskio      0.000000e+00                       25      2024-01-20T21:31:00+00:00         0.000000e+00            0.000000e+00
49           rabbitmq   socket      1.100000e+01                       25      2024-01-20T21:31:00+00:00         1.100000e+01            1.100000e+01
51  rabbitmq-exporter      mem      4.718592e+06                       25      2024-01-20T21:31:00+00:00         4.718592e+06            4.718592e+06
52  rabbitmq-exporter   socket      1.000000e+00                       25      2024-01-20T21:31:00+00:00         1.000000e+00            1.000000e+00
56         session-db   socket      3.000000e+00                       25      2024-01-20T21:31:00+00:00         3.000000e+00            3.000000e+00
73            user-db   socket      1.800000e+01                       25      2024-01-20T21:31:00+00:00         1.800000e+01            1.800000e+01
9            carts-db   socket      9.000000e+00                       20      2024-01-20T21:31:00+00:00         9.000000e+00            9.000000e+00
68               user   socket      3.700000e+01                       11      2024-01-20T21:45:00+00:00         3.700000e+01            3.400000e+01
14          catalogue   socket      9.000000e+00                        5      2024-01-20T21:31:00+00:00         9.400000e+00            8.000000e+00
4               carts   socket      1.400000e+01                        3      2024-01-20T21:31:00+00:00         1.400000e+01            1.326667e+01
64               user   diskio      1.768146e+10                        2      2024-01-20T21:51:00+00:00         1.790394e+10            1.479698e+05
67               user      mem      7.655427e+08                        2      2024-01-20T21:44:00+00:00         8.387559e+08            9.538287e+06
23          front-end      mem      1.064158e+08                        2      2024-01-20T21:32:00+00:00         1.070872e+08            1.037408e+08
60           shipping      mem      3.026157e+08                        2      2024-01-20T21:39:00+00:00         3.031928e+08            3.019515e+08
72            user-db      mem      6.041047e+07                        2      2024-01-20T21:54:00+00:00         6.073060e+07            5.796980e+07, 'P95')```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse loaded 'log' and load error_logs if not present
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps (UTC) for logs and error logs
log["timestamp_dt"] = pd.to_datetime(log["timestamp"], unit="s", utc=True)
error_logs["timestamp_dt"] = pd.to_datetime(error_logs["timestamp"], unit="s", utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-20 21:28:56", tz="UTC")
window_end = pd.Timestamp("2024-01-20 21:58:56", tz="UTC")

# 1) Compute global P95 and P90 per (cmdb_id, log_name) using the full series
log_thresholds = (
    log
    .groupby(["cmdb_id", "log_name"], as_index=False)["value"]
    .agg(p95=lambda x: float(x.quantile(0.95)), p90=lambda x: float(x.quantile(0.90)))
)

# 2) Filter log points to the incident window
log_window = log[log["timestamp_dt"].between(window_start, window_end)]

def compute_log_anomalies(th_col):
    # Merge thresholds into windowed points (thresholds computed from full series)
    lw = log_window.merge(log_thresholds[["cmdb_id", "log_name", th_col]],
                          on=["cmdb_id", "log_name"], how="left")
    if lw.empty:
        return pd.DataFrame(columns=[
            "cmdb_id", "log_name", "global_threshold", "anomaly_count_in_window",
            "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
        ])
    # Anomalies where value >= threshold
    anomalies = lw[lw["value"] >= lw[th_col]]
    if anomalies.empty:
        return pd.DataFrame(columns=[
            "cmdb_id", "log_name", "global_threshold", "anomaly_count_in_window",
            "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
        ])
    anomalies_grp = (
        anomalies
        .groupby(["cmdb_id", "log_name"], as_index=False)
        .agg(
            anomaly_count_in_window=("value", "count"),
            earliest_anomaly_timestamp_utc=("timestamp_dt", "min"),
        )
    )
    # Window stats (max and median among all points in the window for each group)
    window_stats = (
        lw
        .groupby(["cmdb_id", "log_name"], as_index=False)
        .agg(
            max_value_in_window=("value", "max"),
            median_value_in_window=("value", "median"),
        )
    )
    result = (
        anomalies_grp
        .merge(window_stats, on=["cmdb_id", "log_name"], how="left")
        .merge(log_thresholds[["cmdb_id", "log_name", th_col]], on=["cmdb_id", "log_name"], how="left")
    )
    result = result.rename(columns={th_col: "global_threshold"})
    result["earliest_anomaly_timestamp_utc"] = result["earliest_anomaly_timestamp_utc"].apply(lambda x: x.isoformat())
    # Select and order columns
    result = result[[
        "cmdb_id", "log_name", "global_threshold", "anomaly_count_in_window",
        "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
    ]]
    return result

# Try P95 first
result_p95 = compute_log_anomalies("p95")

if not result_p95.empty:
    result_p95 = result_p95.sort_values("anomaly_count_in_window", ascending=False).head(20)
    used_threshold = "P95"
    result_df = result_p95
else:
    # fallback to P90
    result_p90 = compute_log_anomalies("p90")
    if not result_p90.empty:
        result_p90 = result_p90.sort_values("anomaly_count_in_window", ascending=False).head(20)
        used_threshold = "P90"
        result_df = result_p90
    else:
        # no breaches
        result_df = pd.DataFrame(columns=[
            "cmdb_id", "log_name", "global_threshold", "anomaly_count_in_window",
            "earliest_anomaly_timestamp_utc", "max_value_in_window", "median_value_in_window"
        ])
        used_threshold = "none (no P95 or P90 breaches in window)"

# 4) error_logs inside the incident window (up to first 20 rows sorted by timestamp)
error_in_window = error_logs[error_logs["timestamp_dt"].between(window_start, window_end)].sort_values("timestamp_dt").head(20)
if not error_in_window.empty:
    # compact columns and format timestamp to ISO
    error_in_window = error_in_window[["timestamp_dt", "cmdb_id", "message"]].rename(columns={"timestamp_dt": "timestamp"})
    error_in_window["timestamp"] = error_in_window["timestamp"].apply(lambda x: x.isoformat())
else:
    # empty indicator: empty DataFrame with the desired columns
    error_in_window = pd.DataFrame(columns=["timestamp", "cmdb_id", "message"])

# Display compact results
result_df, used_threshold, error_in_window

```
Out[3]:
```
summary = (
    "Log analysis (threshold used: P95):\n"
    "- Many services show log.error_count == P95 == 0 during the incident window (catalogue, user, payment, queue-master, orders, front-end, shipping, carts-db, carts, user-db). "
    "These are zeros equal to the P95 threshold, so they are not strong evidence of increased errors by themselves.\n"
    "- Notable non-zero breaches (compact):\n"
    "  * session-db / log.row_count — global_p95=5.00 — anomalies=5 — earliest=2024-01-20T21:32:00Z — max=5 — median=5\n"
    "  * front-end / log.row_count — global_p95≈2073.80 — anomalies=2 — earliest=2024-01-20T21:42:00Z — max=2088 — median=1956\n"
    "  * catalogue / log.row_count — global_p95≈169.60 — anomalies=2 — earliest=2024-01-20T21:37:00Z — max=171 — median=161\n"
    "  * queue-master / log.row_count — global_p95≈601.20 — anomalies=2 — earliest=2024-01-20T21:32:00Z — max=612 — median=549\n"
    "  * payment / log.row_count — global_p95≈106.80 — anomalies=2 — earliest=2024-01-20T21:32:00Z — max=109 — median=101\n"
    "  * orders / log.row_count — global_p95≈133.60 — anomalies=2 — earliest=2024-01-20T21:32:00Z — max=138 — median=122\n"
    "  * user / log.row_count — global_p95≈628.60 — anomalies=2 — earliest=2024-01-20T21:42:00Z — max=643 — median=582\n"
    "  * shipping / log.row_count — global_p95≈66.80 — anomalies=2 — earliest=2024-01-20T21:32:00Z — max=68 — median=61\n"
    "  * carts-db / log.row_count — global_p95≈1.85 — anomalies=2 — earliest=2024-01-20T21:51:00Z — max=2 — median=1\n"
    "- error_logs: no error_log entries were found inside the incident window (empty result).\n"
    "Interpretation: equality to P95 for log.error_count (zeros) is not strong evidence of errors. The more meaningful signals are spikes in log.row_count for session-db, front-end and several other services during the window, which indicate increased activity or traffic rather than explicit error messages. No error log messages were recorded in the error_logs file during the incident window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  global_threshold  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  median_value_in_window
4      catalogue  log.error_count              0.00                       25      2024-01-20T21:31:00+00:00                    0                     0.0
18          user  log.error_count              0.00                       25      2024-01-20T21:31:00+00:00                    0                     0.0
10       payment  log.error_count              0.00                       25      2024-01-20T21:31:00+00:00                    0                     0.0
12  queue-master  log.error_count              0.00                       25      2024-01-20T21:31:00+00:00                    0                     0.0
8         orders  log.error_count              0.00                       25      2024-01-20T21:31:00+00:00                    0                     0.0
6      front-end  log.error_count              0.00                       25      2024-01-20T21:31:00+00:00                    0                     0.0
16      shipping  log.error_count              0.00                       25      2024-01-20T21:31:00+00:00                    0                     0.0
2       carts-db  log.error_count              0.00                       24      2024-01-20T21:32:00+00:00                    0                     0.0
0          carts  log.error_count              0.00                       20      2024-01-20T21:32:00+00:00                    0                     0.0
20       user-db  log.error_count              0.00                        6      2024-01-20T21:45:00+00:00                    0                     0.0
15    session-db    log.row_count              5.00                        5      2024-01-20T21:32:00+00:00                    5                     5.0
14    session-db  log.error_count              0.00                        5      2024-01-20T21:32:00+00:00                    0                     0.0
3       carts-db    log.row_count              1.85                        2      2024-01-20T21:51:00+00:00                    2                     1.0
5      catalogue    log.row_count            169.60                        2      2024-01-20T21:37:00+00:00                  171                   161.0
13  queue-master    log.row_count            601.20                        2      2024-01-20T21:32:00+00:00                  612                   549.0
11       payment    log.row_count            106.80                        2      2024-01-20T21:32:00+00:00                  109                   101.0
7      front-end    log.row_count           2073.80                        2      2024-01-20T21:42:00+00:00                 2088                  1956.0
9         orders    log.row_count            133.60                        2      2024-01-20T21:32:00+00:00                  138                   122.0
19          user    log.row_count            628.60                        2      2024-01-20T21:42:00+00:00                  643                   582.0
17      shipping    log.row_count             66.80                        2      2024-01-20T21:32:00+00:00                   68                    61.0, 'P95', Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing 'metric' and 'thresholds' if present; otherwise load/compute
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Ensure global thresholds (p95) exist (computed from full series)
try:
    thresholds
except NameError:
    thresholds = (
        metric
        .groupby(["cmdb_id", "kpi_name"], as_index=False)["value"]
        .agg(p95=lambda x: float(x.quantile(0.95)), p90=lambda x: float(x.quantile(0.90)))
    )

# Candidate components
candidates = ['carts','catalogue','orders','payment','user']

# Ensure timestamp_dt exists and is parsed as UTC
if "timestamp_dt" not in metric.columns:
    metric["timestamp_dt"] = pd.to_datetime(metric["timestamp"], unit="s", utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-20 21:28:56", tz="UTC")
window_end = pd.Timestamp("2024-01-20 21:58:56", tz="UTC")

# Compute thresholds for candidate components (global P95 computed from full series)
thr_cand = thresholds[thresholds["cmdb_id"].isin(candidates)][["cmdb_id", "kpi_name", "p95"]]

# Filter metric points to candidate components and the incident window
metric_cand_window = metric[
    metric["cmdb_id"].isin(candidates) & metric["timestamp_dt"].between(window_start, window_end)
].merge(thr_cand, on=["cmdb_id", "kpi_name"], how="left")

# Drop groups without a computed p95 (if any)
metric_cand_window = metric_cand_window[metric_cand_window["p95"].notna()]

# Identify anomalies where value >= global P95
anomalies = metric_cand_window[metric_cand_window["value"] >= metric_cand_window["p95"]]

if anomalies.empty:
    # Empty result (no P95 breaches among candidates)
    result_df = pd.DataFrame(columns=[
        "cmdb_id", "kpi_name", "global_p95", "anomaly_count_in_window",
        "earliest_anomaly_timestamp_utc", "max_value_in_window", "severity_ratio"
    ])
else:
    # Aggregate required stats per (cmdb_id, kpi_name)
    agg_anom = (
        anomalies
        .groupby(["cmdb_id", "kpi_name"], as_index=False)
        .agg(
            anomaly_count_in_window=("value", "count"),
            earliest_anomaly_timestamp_utc=("timestamp_dt", "min"),
            max_value_in_window=("value", "max"),
        )
    )
    # Merge global p95
    agg_anom = agg_anom.merge(thr_cand, on=["cmdb_id", "kpi_name"], how="left").rename(columns={"p95": "global_p95"})
    # Compute severity_ratio per spec
    def compute_severity(row):
        p95 = row["global_p95"]
        mx = row["max_value_in_window"]
        denom = p95 if p95 > 0 else max(1.0, mx)
        return (mx - p95) / denom
    agg_anom["severity_ratio"] = agg_anom.apply(compute_severity, axis=1)
    # Format earliest timestamp as ISO
    agg_anom["earliest_anomaly_timestamp_utc"] = agg_anom["earliest_anomaly_timestamp_utc"].apply(lambda x: x.isoformat())
    # Select columns and sort by severity_ratio desc then anomaly_count desc, limit to 20
    result_df = agg_anom[[
        "cmdb_id", "kpi_name", "global_p95", "anomaly_count_in_window",
        "earliest_anomaly_timestamp_utc", "max_value_in_window", "severity_ratio"
    ]].sort_values(["severity_ratio", "anomaly_count_in_window"], ascending=[False, False]).head(20)

# Display compact result
result_df

```
Out[4]:
```
Summary of metric P95 breaches for the candidate services (carts, catalogue, orders, payment, user):

- Highest relative severity (by severity_ratio):
  1. orders / cpu — global_p95 ≈ 4.34 — anomalies=2 — earliest=2024-01-20T21:39:00Z — max=6.92 — severity_ratio ≈ 0.60
  2. carts / cpu — global_p95 ≈ 2.04 — anomalies=2 — earliest=2024-01-20T21:39:00Z — max=3.16 — severity_ratio ≈ 0.55

- Moderate relative deviations:
  - user / latency-90 — p95 ≈ 0.0145 — anomalies=2 — earliest=2024-01-20T21:51:00Z — max≈0.0178 — severity_ratio ≈ 0.23
  - user / mem — p95 ≈ 7.66e8 — anomalies=2 — earliest=2024-01-20T21:44:00Z — max≈8.39e8 — severity_ratio ≈ 0.10
  - orders / latency-90 and carts / latency-90 — small but present breaches (severity_ratios ≈ 0.08 and 0.045 respectively)

- Lower relative severity but notable counts:
  - catalogue / socket — p95 = 9.0 — anomalies=5 — earliest=2024-01-20T21:31:00Z — max=9.4 — severity_ratio ≈ 0.044 (most frequent breach among listed groups)
  - orders / socket, user / cpu, carts / latency-50, several workload and latency metrics show small >p95 excursions (severity_ratios generally <0.04).

- Other observations:
  - Disk I/O and memory on user show anomalies (user/diskio and user/mem) but with modest severity ratios (user diskio max ≈ 1.79e10 vs p95 ≈ 1.77e10, severity ≈ 0.0126).
  - Most groups that breached had only 2 samples in the window (exceptions: catalogue/socket had 5).

Interpretation / recommended focus:
- Primary attention: CPU spikes on orders and carts (largest severity ratios) during the incident window.
- Secondary: user service shows memory and latency anomalies (worth investigating), and catalogue shows repeated small socket breaches (highest occurrence count).
- These are the groups that exceeded their global P95 within the incident window; groups without rows had no P95 breaches and were omitted.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  severity_ratio
12     orders         cpu  4.339419e+00                        2      2024-01-20T21:39:00+00:00         6.922399e+00        0.595236
0       carts         cpu  2.035317e+00                        2      2024-01-20T21:39:00+00:00         3.163761e+00        0.554432
27       user  latency-90  1.451471e-02                        2      2024-01-20T21:51:00+00:00         1.782638e-02        0.228160
28       user         mem  7.655427e+08                        2      2024-01-20T21:44:00+00:00         8.387559e+08        0.095636
14     orders  latency-90  5.482037e-02                        2      2024-01-20T21:47:00+00:00         5.939459e-02        0.083440
2       carts  latency-90  4.223964e-02                        2      2024-01-20T21:48:00+00:00         4.412953e-02        0.044742
10  catalogue      socket  9.000000e+00                        5      2024-01-20T21:31:00+00:00         9.400000e+00        0.044444
16     orders      socket  1.621333e+01                        2      2024-01-20T21:32:00+00:00         1.690000e+01        0.042352
24       user         cpu  2.344818e+01                        2      2024-01-20T21:51:00+00:00         2.435818e+01        0.038809
1       carts  latency-50  1.922075e-02                        2      2024-01-20T21:48:00+00:00         1.973281e-02        0.026641
17     orders    workload  2.130843e+00                        2      2024-01-20T21:38:00+00:00         2.167717e+00        0.017305
5       carts    workload  8.569177e+00                        2      2024-01-20T21:37:00+00:00         8.716517e+00        0.017194
30       user    workload  1.907038e+01                        2      2024-01-20T21:37:00+00:00         1.939108e+01        0.016817
23    payment    workload  2.174157e+00                        2      2024-01-20T21:38:00+00:00         2.208900e+00        0.015980
25       user      diskio  1.768146e+10                        2      2024-01-20T21:51:00+00:00         1.790394e+10        0.012583
19    payment  latency-50  2.405123e-03                        2      2024-01-20T21:32:00+00:00         2.433296e-03        0.011714
13     orders  latency-50  3.732629e-02                        2      2024-01-20T21:47:00+00:00         3.775635e-02        0.011522
6   catalogue         cpu  1.871834e-01                        2      2024-01-20T21:43:00+00:00         1.883302e-01        0.006127
11  catalogue    workload  4.315570e+00                        2      2024-01-20T21:42:00+00:00         4.333333e+00        0.004116
20    payment  latency-90  4.485587e-03                        2      2024-01-20T21:32:00+00:00         4.500974e-03        0.003430```
```